In [31]:
import re

import pandas as pd

from sklearn.model_selection import train_test_split
from nltk.corpus import stopwords

In [3]:
stop_words = stopwords.words("english")

In [4]:
df1 = pd.read_csv("/home/gpuhead-2/data_mining/DataMiningCourseProject/data/twitter_partisan/cong_politician_tweets_2020-3-12-2021-5-28_text_party_balanced_anonymous.csv")

In [5]:
df1.shape

(2809162, 4)

In [6]:
df2 = pd.read_csv("/home/gpuhead-2/data_mining/DataMiningCourseProject/data/twitter_partisan/nonpolitician_users129_tweets_2021-05-30_anonymous.csv")

In [7]:
df2.shape

(243557, 4)

In [8]:
df1.dropna(subset=["text"], inplace=True)

In [9]:
df2.dropna(subset=["text"], inplace=True)

In [10]:
df2.columns, df1.columns

(Index(['status_id', 'created_at', 'text', 'party'], dtype='str'),
 Index(['status_id', 'created_at', 'text', 'party'], dtype='str'))

In [11]:
df1["source"] = "political"
df2["source"] = "ordinary"

df = pd.concat([df1, df2], ignore_index=True)
df.shape

(3046405, 5)

In [12]:
df = df.dropna(subset=["text", "party"])
df = df.drop_duplicates(subset=["text"])
df.shape

(2588591, 5)

In [13]:
df = df[~df["text"].str.startswith("RT")]
df.shape

(2581320, 5)

In [14]:
df.party.value_counts(normalize=True)

party
D    0.504439
R    0.495561
Name: proportion, dtype: float64

In [15]:
def preprocess(text, remove_stop_words = False):
    """
    preprocess helps clean texts.

    :param text: The target text for cleaning. It should be a string.
    :param remove_stop_words: Whether remove stop words. The defaul is False.

    :return: This function returns cleaned text as a string. 
    """ 

    # remove link
    text = re.sub(r"(http?\://|https?\://|www)\S+", " ", str(text).lower()).strip()
    # remove newlines
    text = re.sub(r'\n', ' ', text)
    # remove puctuations and special characters
    text = re.sub(r'\W+', ' ', text)
    # Substituting multiple spaces with single space
    text = re.sub(r'\s+', ' ', text, flags=re.I)
    # remove first space
    text = re.sub(r'^\s+', '', text)
    # Removing prefixed 'b'
    text = re.sub(r'^b\s+', '', text)
    
    if remove_stop_words:
        tokens = []
        for token in text.split():
            if token not in stop_words:
                tokens.append(token)
        return(" ".join(tokens))
    else:
        return(text)

In [16]:
df.text = df.text.apply(preprocess)

In [19]:
df = df[df.text != ""]

In [30]:
df.groupby("party")["source"].value_counts()

party  source   
D      political    1179550
       ordinary      116734
R      political    1167305
       ordinary      104207
Name: count, dtype: int64

In [41]:
N = 104_000

cells = []

for s in ["political", "ordinary"]:
    for p in ["R", "D"]:
        subset = df[(df.source ==s) & (df.party == p)]
        cells.append(subset.sample(min(len(subset), N), random_state=42))

balanced = pd.concat(cells).sample(frac=1, random_state=42)

stratify_key = balanced["source"] + "_" + balanced["party"]

train_val, test = train_test_split(
    balanced,
    test_size=16000,
    random_state=42,
    stratify=stratify_key
)

In [40]:
balanced.shape

(416000, 5)

In [42]:
train_val.groupby("party")["source"].value_counts()

party  source   
D      political    100000
       ordinary     100000
R      ordinary     100000
       political    100000
Name: count, dtype: int64

In [43]:
test.groupby("party")["source"].value_counts()

party  source   
D      ordinary     4000
       political    4000
R      ordinary     4000
       political    4000
Name: count, dtype: int64

In [44]:
balanced.groupby("party")["source"].value_counts()

party  source   
D      ordinary     104000
       political    104000
R      ordinary     104000
       political    104000
Name: count, dtype: int64

In [45]:
# balanced.to_parquet("datasets/tweets_balanced.parquet", index=False)
balanced.to_csv("/home/gpuhead-2/data_mining/DataMiningCourseProject/data/twitter_partisan/tweets_balanced.csv", index=False)

train_val.to_csv("/home/gpuhead-2/data_mining/DataMiningCourseProject/data/twitter_partisan/tweets_balanced_trainval.csv", index=False)
test.to_csv("/home/gpuhead-2/data_mining/DataMiningCourseProject/data/twitter_partisan/tweets_balanced_test.csv", index=False)

,status_id,created_at,text,party,source
854943,x727864761812135936,2016-05-04 14:17:36,congpalazzo applauded missannmorris of ms who ...,R,political
221770,x790351087647600640,2016-10-24 00:35:57,enemies believe obama is unable or unwilling t...,R,political
989179,x1128003428469415937,2019-05-13 18:25:49,i am deeply sorry for our nation that the hous...,R,political
929396,x1136680674390417409,2019-06-06 17:06:05,the trump admin is quietly forcing mortgage le...,D,political
3014437,x1098766333020381185,2019-02-22 02:08:02,interesting article 5g harmful effects of a ne...,D,ordinary
1353257,x649589216943300608,2015-10-01 14:18:32,,D,political
